# Bengali ASR benchmark
Select a GPU runtime. Upload and extract bengali-asr-benchmark.zip into `/content` first. Use a fresh runtime for each environment profile. No inference has been run in this notebook yet.

In [ ]:
%cd /content/bengali-asr-benchmark
import os, getpass
# Choose standard, speaklar, or indic.
PROFILE = "standard"
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

In [ ]:
import subprocess, sys
if PROFILE == "standard":
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
elif PROFILE == "speaklar":
    assert sys.version_info[:2] in [(3,10),(3,11)], "Speaklar publisher requires Python 3.10/3.11; use a compatible runtime"
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-speaklar.txt"], check=True)
else:
    raise RuntimeError("Install the AI4Bharat NeMo fork in a separate environment following README.md before continuing")

If package installation requests a runtime restart, restart and rerun the directory/token cell, then continue below.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "download_data.py"], check=True)
subprocess.run([sys.executable, "benchmark.py", "prepare"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "-v"], check=True)

In [ ]:
models = {"standard": ["titu-large", "titu-fast"], "speaklar": ["speaklar"], "indic": ["indic"]}[PROFILE]
for model in models:
    subprocess.run([sys.executable, "benchmark.py", "run", "--model", model, "--device", "cuda", "--limit", "8", "--output", "smoke"], check=True)

In [ ]:
for model in models:
    subprocess.run([sys.executable, "benchmark.py", "run", "--model", model, "--device", "cuda"], check=True)

In [ ]:
import pathlib, json
for path in pathlib.Path("results").glob("*/metrics.json"):
    r = json.loads(path.read_text())
    print(r["model"], r["samples"], r["normalized"])
pathlib.Path("results/environment.txt").write_bytes(subprocess.check_output([sys.executable,"-m","pip","freeze"]))